# Serving a Scikit-learn Model with Ray Serve

This notebook trains a gradient-boosting classifier on the Iris dataset and exposes it through Ray Serve. It provides two unauthenticated, local-only ways to submit inference requests:

- A JSON API at `http://127.0.0.1:8000/iris/predict`.
- A browser form at `http://127.0.0.1:8000/iris/`.

Ray's cluster/dashboard token authentication is disabled with `RAY_AUTH_MODE=disabled`. The Ray Dashboard remains bound to `127.0.0.1`, and the Serve proxy is also bound to `127.0.0.1`, so neither interface is exposed to other machines.

> **Security warning:** This configuration is intended only for local learning and development. There is no API key, login, or authorization layer. Do not change either host to `0.0.0.0` or expose the ports through a firewall, tunnel, or public proxy without adding authentication and transport security.

Because authentication mode is selected when Ray starts, restart the notebook kernel before running this notebook if another Ray cluster has already been initialized.

In [ ]:
# Install only if these packages are missing from the selected kernel.
# %pip install "ray[serve]" scikit-learn

## 1. Start a local Ray cluster without dashboard authentication

The environment variable must be set before `ray.init()`. We explicitly bind the Ray Dashboard and Ray Serve HTTP proxy to the loopback interface so disabling authentication does not expose them to the network.

In [ ]:
import os

# Disable Ray cluster/dashboard token authentication for this local example.
os.environ["RAY_AUTH_MODE"] = "disabled"

import ray
from ray import serve

# Clean up a cluster created earlier by this notebook kernel.
if ray.is_initialized():
    serve.shutdown()
    ray.shutdown()

ray_context = ray.init(
    include_dashboard=True,
    dashboard_host="127.0.0.1",
    logging_level="ERROR",
)

serve.start(
    http_options={
        "host": "127.0.0.1",
        "port": 8000,
    }
)

dashboard_address = ray_context.address_info.get("webui_url")
print(f"Ray Dashboard: http://{dashboard_address}")

## 2. Train the model

The Iris dataset contains four numeric features: sepal length, sepal width, petal length, and petal width. The fitted classifier predicts one of three flower species.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import GradientBoostingClassifier

iris_dataset = load_iris()
model = GradientBoostingClassifier(random_state=42)
model.fit(iris_dataset.data, iris_dataset.target)

label_list = iris_dataset.target_names.tolist()
print(f"Classes: {label_list}")

## 3. Define the API and browser request form

FastAPI validates that every request contains exactly four numeric values. `POST /iris/predict` performs inference, while `GET /iris/` serves a small HTML dashboard whose form calls the same prediction endpoint. No authentication middleware or authorization header is required.

In [ ]:
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field


class IrisRequest(BaseModel):
    vector: list[float] = Field(min_length=4, max_length=4)


app = FastAPI(title="Iris Classifier", version="1.0")

DASHBOARD_HTML = """
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Iris Classifier</title>
  <style>
    body { font-family: system-ui, sans-serif; max-width: 720px; margin: 3rem auto; padding: 0 1rem; }
    form { display: grid; grid-template-columns: 1fr 1fr; gap: 1rem; }
    label { display: grid; gap: 0.35rem; font-weight: 600; }
    input, button { padding: 0.65rem; font: inherit; }
    button { grid-column: 1 / -1; cursor: pointer; }
    #result { margin-top: 1.5rem; padding: 1rem; background: #f3f4f6; border-radius: 0.5rem; }
    .error { color: #b91c1c; }
  </style>
</head>
<body>
  <h1>Iris classifier</h1>
  <p>Enter four flower measurements and submit an unauthenticated local inference request.</p>
  <form id="prediction-form">
    <label>Sepal length <input id="sepal-length" type="number" step="any" value="5.1" required></label>
    <label>Sepal width <input id="sepal-width" type="number" step="any" value="3.5" required></label>
    <label>Petal length <input id="petal-length" type="number" step="any" value="1.4" required></label>
    <label>Petal width <input id="petal-width" type="number" step="any" value="0.2" required></label>
    <button type="submit">Predict species</button>
  </form>
  <div id="result">Submit the form to make a prediction.</div>
  <script>
    const form = document.getElementById("prediction-form");
    const result = document.getElementById("result");

    form.addEventListener("submit", async (event) => {
      event.preventDefault();
      const vector = [
        Number(document.getElementById("sepal-length").value),
        Number(document.getElementById("sepal-width").value),
        Number(document.getElementById("petal-length").value),
        Number(document.getElementById("petal-width").value),
      ];

      result.className = "";
      result.textContent = "Predicting...";

      try {
        const response = await fetch("predict", {
          method: "POST",
          headers: { "Content-Type": "application/json" },
          body: JSON.stringify({ vector }),
        });
        const body = await response.json();
        if (!response.ok) {
          throw new Error(JSON.stringify(body));
        }
        result.textContent = `Prediction: ${body.result} (${(body.confidence * 100).toFixed(1)}% confidence)`;
      } catch (error) {
        result.className = "error";
        result.textContent = `Request failed: ${error.message}`;
      }
    });
  </script>
</body>
</html>
"""


@serve.deployment
@serve.ingress(app)
class BoostingModel:
    def __init__(self, fitted_model, labels):
        self.model = fitted_model
        self.labels = labels

    @app.get("/", response_class=HTMLResponse)
    async def dashboard(self):
        return DASHBOARD_HTML

    @app.get("/health")
    async def health(self):
        return {"status": "ok"}

    @app.post("/predict")
    async def predict(self, payload: IrisRequest):
        prediction = int(self.model.predict([payload.vector])[0])
        probabilities = self.model.predict_proba([payload.vector])[0]
        return {
            "result": self.labels[prediction],
            "confidence": float(probabilities[prediction]),
            "vector": payload.vector,
        }

## 4. Deploy and open the interfaces

The Ray Dashboard shows cluster, actor, and Serve deployment information. The model dashboard is the browser form used to submit inference requests.

In [ ]:
serve.run(
    BoostingModel.bind(model, label_list),
    name="iris-classifier",
    route_prefix="/iris",
)

print(f"Ray Dashboard: http://{dashboard_address}")
print("Model dashboard: http://127.0.0.1:8000/iris/")
print("Prediction API: http://127.0.0.1:8000/iris/predict")
print("Health check: http://127.0.0.1:8000/iris/health")

## 5. Test the unauthenticated API

This request contains no token, API key, cookie, or authorization header. A successful response confirms that the model endpoint accepts unauthenticated local requests.

In [ ]:
import requests

sample_request = {"vector": [5.1, 3.5, 1.4, 0.2]}
response = requests.post(
    "http://127.0.0.1:8000/iris/predict",
    json=sample_request,
    timeout=10,
)
response.raise_for_status()
print(response.json())

## 6. Stop the service

Run the following cell when you are finished. The HTTP endpoint and both dashboards remain available only while Ray Serve and the local Ray cluster are running.

In [ ]:
# Uncomment when you are ready to stop the deployment and local cluster.
# serve.shutdown()
# ray.shutdown()

## Conclusion

The Iris classifier is served through a validated JSON API and a browser-based request form. Ray's dashboard authentication and the model endpoint's authentication are both disabled for local development, while loopback-only binding prevents network access from other machines.

For production, keep Ray's control-plane authentication enabled, place the inference service behind an authenticated reverse proxy or API gateway, use HTTPS, restrict network access, and add request-rate and payload-size limits.